###Create Validation Views

This cell creates unified temporary views for Silver and Bronze data to compare and validate records.

In [0]:
-- Creates unified temporary views for Silver and Bronze data to compare and validate records.

CREATE OR REPLACE TEMP VIEW v_silver_all AS
SELECT 'blogs' AS tbl, content_id, url, topic, category, CAST(NULL AS STRING) AS difficulty_level, published_date, last_updated FROM edtech.silver.blogs
UNION ALL SELECT 'newsletters',      content_id, url, topic, category, NULL,             published_date, last_updated FROM edtech.silver.newsletters
UNION ALL SELECT 'coursera_courses', content_id, url, topic, category, difficulty_level, published_date, last_updated FROM edtech.silver.coursera_courses
UNION ALL SELECT 'microsoft_learn',  content_id, url, topic, category, difficulty_level, published_date, last_updated FROM edtech.silver.microsoft_learn
UNION ALL SELECT 'github_repos',     content_id, url, topic, category, difficulty_level, published_date, last_updated FROM edtech.silver.github_repos
UNION ALL SELECT 'youtube_videos',   content_id, url, topic, category, difficulty_level, published_date, last_updated FROM edtech.silver.youtube_videos;

-- Maps each Bronze row to its Silver table. Rows that match nothing get tbl = NULL.
CREATE OR REPLACE TEMP VIEW v_bronze_all AS
SELECT CASE LOWER(TRIM(content_type))
         WHEN 'article'    THEN 'blogs'
         WHEN 'newsletter' THEN 'newsletters' END AS tbl,
       content_id, title, url
FROM edtech.bronze.rss_raw
UNION ALL
SELECT CASE LOWER(TRIM(source))
         WHEN 'coursera'        THEN 'coursera_courses'
         WHEN 'microsoft learn' THEN 'microsoft_learn'
         WHEN 'github'          THEN 'github_repos'
         WHEN 'youtube'         THEN 'youtube_videos' END,
       content_id, title, url
FROM edtech.bronze.api_raw;

###Check Bronze vs Silver



In [0]:
-- Compares valid Bronze records with Silver records and identifies data gaps.

SELECT b.tbl, 
       b.bronze_rows, 
       b.bronze_valid_ids, 
       s.silver_rows, 
       b.bronze_valid_ids - s.silver_rows AS unexplained_gap 
FROM ( 
  -- Counts total and valid Bronze records
  SELECT tbl, 
         COUNT(*) AS bronze_rows, 
         COUNT(DISTINCT CASE WHEN NULLIF(TRIM(content_id),'') IS NOT NULL 
                              AND NULLIF(TRIM(title),'')      IS NOT NULL 
                              AND NULLIF(TRIM(url),'')        IS NOT NULL 
                             THEN TRIM(content_id) END) AS bronze_valid_ids 
  FROM v_bronze_all GROUP BY tbl 
) b 
LEFT JOIN (
  -- Counts records in Silver
  SELECT tbl, COUNT(*) AS silver_rows 
  FROM v_silver_all GROUP BY tbl
) s 
  ON b.tbl = s.tbl 
ORDER BY b.tbl;

tbl,bronze_rows,bronze_valid_ids,silver_rows,unexplained_gap
blogs,565,565,565,0
coursera_courses,300,300,300,0
github_repos,300,300,300,0
microsoft_learn,300,300,300,0
newsletters,35,35,35,0
youtube_videos,300,300,300,0


###Check Duplicate Content IDs



In [0]:
-- Uniqueness check: fails the cell immediately if any table has duplicate content_id.
SELECT assert_true(
  (SELECT MAX(dup) FROM (
     SELECT COUNT(*) - COUNT(DISTINCT content_id) AS dup
     FROM v_silver_all GROUP BY tbl
   )) = 0,
  'Duplicate content_id found in at least one Silver table'
) AS duplicate_content_ids_check;

duplicate_content_ids_check
null


### Check Duplicate URLs



In [0]:
-- Uniqueness check: fails if any table has duplicate url values.
SELECT assert_true(
  (SELECT MAX(dup) FROM (
     SELECT COUNT(*) - COUNT(DISTINCT url) AS dup
     FROM v_silver_all GROUP BY tbl
   )) = 0,
  'Duplicate url found in at least one Silver table'
) AS duplicate_urls_check;

duplicate_urls_check
null


### Check Bad URLs



In [0]:
-- Validity check: fails if any url does not start with http.
SELECT assert_true(
  (SELECT SUM(CASE WHEN url NOT LIKE 'http%' THEN 1 ELSE 0 END) FROM v_silver_all) = 0,
  'Malformed url found (does not start with http)'
) AS bad_urls_check;

bad_urls_check
null


### Check Date Range


In [0]:
-- Validity check: fails if any published_date is in the future or before year 2000.
SELECT assert_true(
  (SELECT SUM(CASE WHEN published_date > current_timestamp()
                      OR published_date < TIMESTAMP '2000-01-01'
                    THEN 1 ELSE 0 END)
   FROM v_silver_all) = 0,
  'published_date out of sane range found'
) AS out_of_range_dates_check;

out_of_range_dates_check
null


### Check Bronze to Silver Gap



In [0]:
-- Completeness check: fails if the count of valid Bronze rows doesn't match Silver rows.
SELECT assert_true(
  (SELECT MAX(ABS(gap)) FROM (
     SELECT b.tbl, b.bronze_valid_ids - COALESCE(s.silver_rows, 0) AS gap
     FROM (
       SELECT tbl,
              COUNT(DISTINCT CASE WHEN NULLIF(TRIM(content_id),'') IS NOT NULL
                                    AND NULLIF(TRIM(title),'')     IS NOT NULL
                                    AND NULLIF(TRIM(url),'')       IS NOT NULL
                                   THEN TRIM(content_id) END) AS bronze_valid_ids
       FROM v_bronze_all GROUP BY tbl
     ) b
     LEFT JOIN (
       SELECT tbl, COUNT(*) AS silver_rows FROM v_silver_all GROUP BY tbl
     ) s ON b.tbl = s.tbl
   )) = 0,
  'Unexplained gap between Bronze valid rows and Silver rows'
) AS bronze_to_silver_gap_check;

bronze_to_silver_gap_check
null


### Check for NaN Keywords


In [0]:
-- Validity check: fails if any 3-tier keyword extraction left the literal string "NaN".
SELECT assert_true(
  (SELECT SUM(nan_count) FROM (
     SELECT SUM(CASE WHEN list_of_keywords = 'NaN' THEN 1 ELSE 0 END) AS nan_count FROM edtech.silver.github_repos
     UNION ALL
     SELECT SUM(CASE WHEN list_of_keywords = 'NaN' THEN 1 ELSE 0 END) FROM edtech.silver.microsoft_learn
     UNION ALL
     SELECT SUM(CASE WHEN list_of_keywords = 'NaN' THEN 1 ELSE 0 END) FROM edtech.silver.youtube_videos
   )) = 0,
  'Leftover "NaN" string found in list_of_keywords'
) AS nan_keywords_check;

nan_keywords_check
null


### Check Missing Published Dates



In [0]:
-- Completeness check (warning only, not a hard gate): shows count of missing
-- published_date per table. Not wrapped in assert_true on purpose — a missing
-- date may be a real source-feed gap, not a pipeline bug, so we just observe it.
SELECT tbl, SUM(CASE WHEN published_date IS NULL THEN 1 ELSE 0 END) AS null_published
FROM v_silver_all
GROUP BY tbl;

tbl,null_published
blogs,0
newsletters,0
coursera_courses,0
microsoft_learn,0
github_repos,0
youtube_videos,0
